# VULCAN posterior uncertainty vs repeated-scan variance

It does not import the SPICE project code. It uses only `numpy` and `matplotlib`.



## CELL 1: load data

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

HERE = Path.cwd()
if not (HERE / 'data' / 'metadata.json').exists():
    HERE = Path('review_vulcan_mc_notebook')
DATA = HERE / 'data'

with open(DATA / 'metadata.json', 'r') as f:
    meta = json.load(f)

RUN_TAGS = meta['run_tags']
SUBJECTS = meta['subjects']
NY, NX = meta['dim']
NSEQ = meta['n_seq']
ppm_axis = np.load(DATA / 'ppm_axis.npy')
wref_norm = np.load(DATA / 'wref_norm.npy')
brain_mask = np.load(DATA / 'brain_mask.npy')

def tag_key(run_tag):
    return run_tag.replace('.', 'p').replace('-', 'm')

def fid_to_spec(fid):
    return np.fft.fftshift(np.fft.fft(fid, axis=-1, norm='ortho'), axes=-1)

def mean_uncert_map(std):
    return np.mean(np.abs(std), axis=-1)

def masked_values(arr):
    vals = arr[brain_mask]
    return vals[np.isfinite(vals)]

def rankdata_average(x):
    x = np.asarray(x)
    order = np.argsort(x, kind='mergesort')
    ranks = np.empty(len(x), dtype=float)
    sx = x[order]
    starts = np.r_[0, np.flatnonzero(sx[1:] != sx[:-1]) + 1]
    ends = np.r_[starts[1:], len(x)]
    for start, end in zip(starts, ends):
        ranks[order[start:end]] = 0.5 * (start + end - 1) + 1.0
    return ranks

def spearman_corr(x, y):
    rx = rankdata_average(x)
    ry = rankdata_average(y)
    return float(np.corrcoef(rx, ry)[0, 1])

print('Data folder:', DATA)
print('Subjects:', SUBJECTS)
print('Run tags:', RUN_TAGS)
print('Brain voxels:', int(brain_mask.sum()))

## 1. Repeated-scan variance calculation

For each lambda, the package stores the five repeat reconstructions as:

```text
spice_fid_stack_<lambda>.npy  shape = (5, 64, 64, 300)
```

The repeated-scan uncertainty is computed directly from these five reconstructions. Each `SPICE_f` array is in the FID/time domain. We first apply a unitary FFT along the spectral dimension to obtain spectra, then compute the standard deviation across the five repeated scans at each voxel and ppm:

```python
# spice_fid_stack: (n_scans, Ny, Nx, N_time)
spec_stack = fftshift(fft(spice_fid_stack, axis=-1, norm='ortho'), axes=-1)

# repeated-scan uncertainty: (Ny, Nx, N_ppm)
mc_std = std(spec_stack, axis=0)
```

So the variance is across repeat scans at fixed voxel and fixed ppm.

In [ ]:
run_tag = RUN_TAGS[0]
key = tag_key(run_tag)

spice_fid_stack = np.load(DATA / f'spice_fid_stack_{key}.npy')
spec_stack = fid_to_spec(spice_fid_stack)
mc_std_from_stack = np.std(spec_stack, axis=0)
mc_std_saved = np.load(DATA / f'mc_std_{key}.npy')

relerr = np.linalg.norm(mc_std_from_stack - mc_std_saved) / (np.linalg.norm(mc_std_saved) + 1e-30)
print('run_tag:', run_tag)
print('SPICE_f repeat stack:', spice_fid_stack.shape, spice_fid_stack.dtype)
print('MC std:', mc_std_from_stack.shape, mc_std_from_stack.dtype)
print('relative error vs saved mc_std:', relerr)

plt.figure(figsize=(5, 4))
plt.imshow(np.where(brain_mask, mean_uncert_map(mc_std_from_stack), np.nan), origin='lower', cmap='magma')
plt.colorbar(label='mean spectral std')
plt.title(f'Repeated-scan MC std map\n{run_tag}')
plt.tight_layout()

## 2. VULCAN posterior uncertainty from mHm

The posterior covariance at each voxel is built from the Hessian inverse block:

```text
Sigma_v = 2 * sigma_noise^2 * V @ mHm_v @ V^H
```

Here `V` is the saved SPICE subspace in the FID/time domain, so `V @ mHm_v @ V^H` is also an FID-domain covariance. No `spec_to_fid` conversion is applied here. In the original VULCAN post-processing, posterior samples are drawn in FID domain and then transformed to spectrum domain with the same unitary FFT used above.

The full `Sigma_v` is a 300 x 300 complex covariance matrix for every voxel, which is too large for a lightweight review folder. This package therefore stores:

- `vulcan_std_<lambda>.npy`: the final VULCAN posterior spectral std used in the comparison.
- `vmhmv_diag_fid_<lambda>.npy`: only the diagonal of the FID-domain covariance `2 sigma_noise^2 V mHm V^H`. This is a lightweight Hessian-scale diagnostic, not the plotted spectral covariance.

In [ ]:
vulcan_std = np.load(DATA / f'vulcan_std_{key}.npy')
vmhmv_diag_fid = np.load(DATA / f'vmhmv_diag_fid_{key}.npy')

print('VULCAN posterior std:', vulcan_std.shape, vulcan_std.dtype)
print('diag(2 sigma^2 V mHm V^H), FID-domain diagnostic:', vmhmv_diag_fid.shape, vmhmv_diag_fid.dtype)
print('median VULCAN mean std in brain:', float(np.nanmedian(mean_uncert_map(vulcan_std)[brain_mask])))
print('median sqrt vmhmv_diag mean in brain:', float(np.nanmedian(np.sqrt(np.maximum(vmhmv_diag_fid, 0)).mean(axis=-1)[brain_mask])))

plt.figure(figsize=(5, 4))
plt.imshow(np.where(brain_mask, mean_uncert_map(vulcan_std), np.nan), origin='lower', cmap='magma')
plt.colorbar(label='mean spectral std')
plt.title(f'VULCAN posterior std map\n{run_tag}')
plt.tight_layout()

## 3. Lambda selector: maps, histogram, and scatter

The scatter plot uses one point per `brain voxel x ppm bin`:

```text
x = VULCAN_std[voxel, ppm]
y = repeated_scan_MC_std[voxel, ppm]
```

The ratio map/histogram uses one value per voxel:

```text
ratio(voxel) = mean_ppm(MC_std) / mean_ppm(VULCAN_std)
```

In [ ]:
def show_run(run_tag):
    key = tag_key(run_tag)
    mc_std = np.load(DATA / f'mc_std_{key}.npy')
    vulcan_std = np.load(DATA / f'vulcan_std_{key}.npy')

    mc_map = mean_uncert_map(mc_std)
    vu_map = mean_uncert_map(vulcan_std)
    ratio_map = mc_map / (vu_map + 1e-30)
    ratio_vals = masked_values(ratio_map)

    brain3d = np.broadcast_to(brain_mask[:, :, None], mc_std.shape)
    x = np.abs(vulcan_std)[brain3d]
    y = np.abs(mc_std)[brain3d]
    ok = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x = x[ok]
    y = y[ok]
    slope = float((x @ y) / (x @ x))
    spearman_r = spearman_corr(x, y)

    rng = np.random.default_rng(0)
    if len(x) > 5000:
        idx = rng.choice(len(x), 5000, replace=False)
        xs, ys = x[idx], y[idx]
    else:
        xs, ys = x, y

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.ravel()

    def draw_map(ax, arr, title, cmap='magma', vmin=None, vmax=None, norm=None):
        shown = np.where(brain_mask, arr, np.nan)
        im = ax.imshow(shown, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax, norm=norm)
        ax.imshow(np.where(brain_mask, wref_norm, np.nan), origin='lower', cmap='gray', alpha=0.25)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046)

    vmax_uncert = np.nanpercentile(np.r_[masked_values(mc_map), masked_values(vu_map)], 97.5)
    draw_map(axes[0], mc_map, 'Repeated-scan MC std\nmean over ppm', vmax=vmax_uncert)
    draw_map(axes[1], vu_map, 'VULCAN posterior std\nmean over ppm', vmax=vmax_uncert)

    lo, hi = np.nanpercentile(ratio_vals, [2.5, 97.5])
    lo = min(lo, 0.9)
    hi = max(hi, 1.1)
    ratio_norm = TwoSlopeNorm(vmin=lo, vcenter=1.0, vmax=hi)
    draw_map(axes[2], ratio_map, 'Ratio map: MC / VULCAN\nwhite = 1.0', cmap='RdBu_r', norm=ratio_norm)

    axes[3].hist(ratio_vals, bins=80, color='steelblue', alpha=0.85)
    axes[3].axvline(np.nanmedian(ratio_vals), color='tomato', label=f'median={np.nanmedian(ratio_vals):.3f}')
    axes[3].axvline(1.0, color='limegreen', linestyle='--', label='ratio=1')
    axes[3].set_title('Voxel ratio histogram')
    axes[3].set_xlabel('mean_ppm(MC) / mean_ppm(VULCAN)')
    axes[3].legend()

    axes[4].scatter(xs, ys, s=4, alpha=0.35, color='steelblue', rasterized=True)
    lim = np.nanpercentile(np.r_[x, y], 99.0)
    axes[4].plot([0, lim], [0, lim], color='limegreen', linestyle='--', linewidth=1, label='y=x')
    axes[4].plot([0, lim], [0, slope * lim], color='tomato', linewidth=1, label=f'origin slope={slope:.3f}')
    axes[4].set_xlim(0, lim)
    axes[4].set_ylim(0, lim)
    axes[4].set_xlabel('VULCAN std')
    axes[4].set_ylabel('Repeated-scan MC std')
    axes[4].set_title(f'Scatter: brain voxels x ppm\nSpearman r={spearman_r:.3f}, n={len(x)}')
    axes[4].legend()

    axes[5].axis('off')
    text = (
        f'run tag: {run_tag}\n'
        f'brain voxels: {int(brain_mask.sum())}\n'
        f'median ratio: {np.nanmedian(ratio_vals):.4f}\n'
        f'mean ratio: {np.nanmean(ratio_vals):.4f}\n'
        f'IQR ratio: {np.nanpercentile(ratio_vals, 75) - np.nanpercentile(ratio_vals, 25):.4f}\n'
        f'origin scatter slope: {slope:.4f}\n'
        f'Spearman scatter r: {spearman_r:.4f}'
    )
    axes[5].text(0.0, 0.95, text, va='top', family='monospace', fontsize=12)

    fig.suptitle(f'Repeated-scan MC vs VULCAN posterior uncertainty: {run_tag}', fontsize=14)
    plt.tight_layout()
    plt.show()

# Manually choose one of:
#   'w5000_l0.0001'
#   'w5000_l1e-05'
#   'w5000_l1e-06'
run_tag = 'w5000_l0.0001'

show_run(run_tag)